In [8]:
# Library for dataframe reading
import pandas as pd

# Library for visualizations 
import seaborn as sns

# Library for SQL queries
import getpass

import pg8000


In [4]:
# connecting to the database 
login = input('Login username: ')
secret = getpass.getpass()

credentials = {'user'    : login,
               'password': secret, 
               'database': 'csci403',
               'host'    : 'ada.mines.edu'}

db = pg8000.connect(**credentials)

cursor = db.cursor()

NameError: name 'getpass' is not defined

In [3]:
# DATA EXPLORATION: what is the average trip duration for each start and end station pair in the dataset?

# NOTE: before performing the data aggregation, there was a view created to obtain a 
# subset of the data (see in queries.sql)

# perform data aggregation
average_trip_query = """
SELECT start_station_name,
end_station_name,
usertype,
st_month,
st_hour,
ed_hour,
gender,
start_dpcapactiy,
end_dpcapacity ,
       AVG(trip_duration)
FROM subset_trip
WHERE end_station_name IS NOT NULL 
AND start_station_name IS NOT NULL
AND usertype IS NOT NULL
AND st_month IS NOT NULL
AND ed_hour IS NOT NULL
AND start_dpcapactiy IS NOT NULL
AND end_dpcapacity IS NOT NULL 
AND gender <> -1
GROUP BY usertype, st_month, st_hour, ed_hour, start_dpcapactiy,  end_dpcapacity, start_station_name, gender, end_station_name
ORDER BY end_station_name

"""
cursor.execute(average_trip_query)

# fetch results from aggregation
average_trip = cursor.fetchall()

NameError: name 'cursor' is not defined

In [18]:
# convert list to visual dataframe (Easier for reading exact data)
average_trip = pd.DataFrame(average_trip, columns=["start_station_name","end_station_name","usertype","st_month","st_hour","ed_hour","gender","start_dpcapactiy","end_dpcapactiy","avg_trip_duration"])

In [22]:
# converting avg to a numeric column
average_trip.info()
average_trip["avg_trip_duration"] = pd.to_numeric(average_trip['avg_trip_duration'])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36402 entries, 0 to 36401
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   start_station_name  36402 non-null  object
 1   end_station_name    36402 non-null  object
 2   usertype            36402 non-null  int64 
 3   st_month            36402 non-null  int64 
 4   st_hour             36402 non-null  int64 
 5   ed_hour             36402 non-null  int64 
 6   gender              36402 non-null  int64 
 7   start_dpcapactiy    36402 non-null  int64 
 8   end_dpcapactiy      36402 non-null  int64 
 9   avg_trip_duration   36402 non-null  object
dtypes: int64(7), object(3)
memory usage: 2.8+ MB


In [23]:
# Checking to see if conversion was successful
average_trip.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36402 entries, 0 to 36401
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   start_station_name  36402 non-null  object 
 1   end_station_name    36402 non-null  object 
 2   usertype            36402 non-null  int64  
 3   st_month            36402 non-null  int64  
 4   st_hour             36402 non-null  int64  
 5   ed_hour             36402 non-null  int64  
 6   gender              36402 non-null  int64  
 7   start_dpcapactiy    36402 non-null  int64  
 8   end_dpcapactiy      36402 non-null  int64  
 9   avg_trip_duration   36402 non-null  float64
dtypes: float64(1), int64(7), object(2)
memory usage: 2.8+ MB


In [ ]:
# NOTE: creating a heat-map to show which commute (to and from) are the longest

import matplotlib.pyplot as plt
# pivot into matrix form
pivot = average_trip.pivot(index='start', columns='end', values='avg_trip_duration')

# optional: reduce size (top N most common starts/ends)
top_starts = average_trip['start'].value_counts().head(20).index
top_ends = average_trip['end'].value_counts().head(20).index
pivot = pivot.loc[top_starts, top_ends]

plt.figure(figsize=(12,8))
sns.heatmap(pivot, cmap='viridis')
plt.title("Average Duration (Start → End)")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()

In [ ]:
# DATA EXPLORATION: what is the average trip duration by gender?

avg_trip_gender_query = """
SELECT start_station_name,
gender,
end_station_name,
       AVG(trip_duration)
FROM subset_trip
WHERE end_station_name IS NOT NULL 
AND start_station_name IS NOT NULL
AND end_station_name is NOT NULL
GROUP BY start_station_name, end_station_name
ORDER BY end_station_name
                        """
cursor.execute(avg_trip_gender_query)

average_trip_gender = cursor.fetchall()

average_trip_gender = pd.DataFrame(average_trip, columns=["start_location","end_location","avg_trip_duration","gender"])
average_trip_gender["avg"] = pd.to_numeric(average_trip['avg'])

In [ ]:
average_trip_gender.head(10000)

In [ ]:
df = average_trip_gender.copy()

# make sure numeric
df['avg_trip_duration'] = pd.to_numeric(df['avg_trip_duration'], errors='coerce')
df = df.dropna(subset=['avg_trip_duration'])

# make gender readable
df['gender_label'] = df['gender'].map({0: 'Female', 1: 'Male'})


In [ ]:
# reduce size so plot is readable
top_starts = df['start_location'].value_counts().head(15).index
top_ends = df['end_location'].value_counts().head(15).index

df_small = df[
    df['start_location'].isin(top_starts) &
    df['end_location'].isin(top_ends)
]

genders = df_small['gender_label'].dropna().unique()

fig, axes = plt.subplots(1, len(genders), figsize=(16,6), sharey=True)

# handle case where only one gender exists
if len(genders) == 1:
    axes = [axes]

for ax, gender in zip(axes, genders):
    subset = df_small[df_small['gender_label'] == gender]
    
    pivot = subset.pivot(
        index='start_location',
        columns='end_location',
        values='avg_trip_duration'
    )
    
    sns.heatmap(pivot, cmap='viridis', ax=ax)
    ax.set_title(gender)

plt.tight_layout()
plt.show()

Research Question: Is there a difference in average trip duartion for each start and end location pair between male and female Divvy bike users? 

$H_0$: For any given start–end pair, males and females have the same average trip duration.
$H_a$: For at least some routes, males and females have different average trip durations.

In [ ]:
pivot = df.pivot_table(
    index=['start_location','end_location'],
    columns='gender_label',
    values='avg_trip_duration'
).dropna()

diff = pivot['Female'] - pivot['Male']

from scipy.stats import ttest_1samp
t_stat, p_value = ttest_1samp(diff, 0)

print("t-stat:", t_stat)
print("p-value:", p_value)

In [ ]:
# Distribution of Female Avg trip duration
pivot['Female']
sns.histplot(pivot['Female'])

In [ ]:
# Distribution of male trip duration
sns.histplot(pivot['Male'])

Conclusion: There is statistically significant evidence that for at least some routes in the data, males and females have different trip averages. 

Reasearch Question: Is gender a good predictor of the average time between specific stations in Chicago?

Research Question: Is user type (casual or member) a good predictor of the average time between specific stations in chicago?

Research Question: is the docking point capacity of the start station a good predictor of the average time between specific stations in Chicago? what about the end station?


In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    'avg_trip_duration ~ C(start_location) + C(end_location) + C(gender)',
    data=average_trip_gender
).fit()

print(model.summary())

In [ ]:
from scipy.stats import f_oneway

groups = [
    group['avg_trip_duration'].values
    for _, group in df.groupby('gender_label')
]

f_stat, p_val = f_oneway(*groups)

In [ ]:
p_val

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

df = average_trip_gender.copy()

# features + target
X = df[['start_location', 'end_location', 'gender']]
y = df['avg_trip_duration']

# preprocessing (encode categories)
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['start_location','end_location','gender'])
    ]
)

model = Pipeline([
    ('prep', preprocess),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model.fit(X_train, y_train)

preds = model.predict(X_test)

print("RMSE:", mean_squared_error(y_test, preds, squared=False))

In [ ]:
df['avg_trip_duration'].mean()

In [ ]:
sns.histplot(df['avg_trip_duration'])

The mean trip time for each Only using start location, end location, and gender is not enough to predict the average trip duration between stations in Chicago.

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = average_trip.copy()

df['route'] = df['start_station_name'] + " → " + df['end_station_name']

model = smf.ols(
    'avg_trip_duration ~ C(route) + C(usertype) + C(st_month) + C(st_hour) + C(ed_hour) + C(gender) + start_dpcapactiy + end_dpcapactiy',
    data=df
).fit()

In [28]:
from statsmodels.stats.anova import anova_lm

anova_results = anova_lm(model)
print(anova_results)

                            df        sum_sq       mean_sq           F  \
C(start_station_name)    299.0  6.014818e+08  2.011645e+06    8.379444   
C(end_station_name)      298.0  5.220654e+08  1.751897e+06    7.297474   
C(usertype)                0.0  0.000000e+00           NaN         NaN   
C(st_month)                0.0  0.000000e+00           NaN         NaN   
C(st_hour)                23.0  4.731198e+07  2.057043e+06    8.568547   
C(ed_hour)                23.0  7.554141e+08  3.284409e+07  136.811037   
C(gender)                  1.0  4.967741e+07  4.967741e+07  206.929714   
start_dpcapactiy           1.0  3.325164e+04  3.325164e+04    0.138509   
end_dpcapactiy             1.0  1.031756e+05  1.031756e+05    0.429775   
Residual               35757.0  8.584148e+09  2.400690e+05         NaN   

                              PR(>F)  
C(start_station_name)   0.000000e+00  
C(end_station_name)    1.239571e-271  
C(usertype)                      NaN  
C(st_month)                  

In [30]:
reduced_model = smf.ols(
    'avg_trip_duration ~ C(start_station_name) + C(end_station_name) + C(st_hour) + C(ed_hour) + C(gender) + start_dpcapactiy + end_dpcapactiy',
    data=df
).fit()

In [31]:
anova_lm(reduced_model,model)

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,35757.0,8.584148e+09,0.0,NaN,NaN,NaN
1,35757.0,8.584148e+09,-0.0,-0.0,NaN,NaN


In [2]:
df = average_trip.copy()

NameError: name 'average_trip' is not defined